# Flow Matching: Equation of State Fitting

Fit van der Waals equation parameters $(a, b)$ from observed pressure data.

**Authors:** Victor Alves and John R. Kitchin

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
import os

# Force CPU for JAX (must be set before importing JAX)
os.environ['JAX_PLATFORMS'] = 'cpu'

import torch

# Import reusable utilities from local module
from generative_optimization import (
    generate_samples,
    cluster_stats,
    ConditionalFlowMatching
)

# Force CPU for PyTorch
device = torch.device('cpu')
print(f"Using device: {device}")

# Figure settings
mpl.rcParams['figure.facecolor'] = 'white'
mpl.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['figure.dpi'] = 150

warnings.filterwarnings('ignore')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Problem Setup

Van der Waals equation:
$$P = \frac{RT}{V - b} - \frac{a}{V^2}$$

Given pressure measurements at different $(V, T)$ conditions, estimate $(a, b)$.

In [ ]:
# Van der Waals equation
R_gas = 8.314  # J/(mol·K)

def vdw_pressure(V, T, a, b):
    """Compute pressure from van der Waals equation."""
    return R_gas * T / (V - b) - a / V**2

# True parameters (for CO2-like gas)
a_true = 0.364  # Pa·m⁶/mol²
b_true = 4.27e-5  # m³/mol

print(f"True parameters: a = {a_true:.4f} Pa·m⁶/mol², b = {b_true:.2e} m³/mol")

In [ ]:
# Generate training data: sample (a, b) and compute pressures at representative conditions
n_samples = 3000
np.random.seed(42)

# Sample parameters around reasonable range
a_samples = np.random.uniform(0.1, 0.8, n_samples)
b_samples = np.random.uniform(1e-5, 1e-4, n_samples)

# Representative V, T values for computing pressures
V_repr = np.array([1.5e-4, 2.5e-4, 3.5e-4])  # m³/mol
T_repr = np.array([350, 400, 450])  # K

# Compute pressures for all parameter combinations
P_samples = []
for a, b in zip(a_samples, b_samples):
    P_vals = []
    for V, T in zip(V_repr, T_repr):
        try:
            P = vdw_pressure(V, T, a, b)
            P_vals.append(P / 1e6)  # Convert to MPa
        except:
            P_vals.append(np.nan)
    P_samples.append(P_vals)

P_samples = np.array(P_samples)

# Remove invalid samples (negative pressure, etc.)
valid = ~np.isnan(P_samples).any(axis=1) & (P_samples > 0).all(axis=1)
a_samples = a_samples[valid]
b_samples = b_samples[valid]
P_samples = P_samples[valid]

print(f"Valid samples: {len(a_samples)}")
print(f"P range: [{P_samples.min():.2f}, {P_samples.max():.2f}] MPa")

In [ ]:
# Train Flow Matching: generate [a, b] conditioned on [P1, P2, P3]
# Scale b by 1e4 for better numerical conditioning
x_data = np.column_stack([a_samples, b_samples * 1e4])  # Parameters to generate
c_data = P_samples  # Pressures to condition on

fm_eos = ConditionalFlowMatching(x_dim=2, c_dim=3, hidden_dim=128, n_layers=4)
losses = fm_eos.fit(x_data, c_data, epochs=1000, batch_size=64)

In [ ]:
# Inverse problem: given observed pressures, find (a, b)
# Compute pressures at representative points using true parameters
P_obs = [vdw_pressure(V, T, a_true, b_true) / 1e6 for V, T in zip(V_repr, T_repr)]
print(f"Observed pressures (MPa): {[f'{p:.3f}' for p in P_obs]}")

# Sample parameters conditioned on observed pressures
samples = fm_eos.sample(c_values=[P_obs], n_samples=1000, n_steps=100)

# Unscale b
a_est = samples[:, 0]
b_est = samples[:, 1] * 1e-4

print(f"\nEstimated parameters:")
print(f"  a = {a_est.mean():.4f} ± {a_est.std():.4f} (true: {a_true:.4f})")
print(f"  b = {b_est.mean():.2e} ± {b_est.std():.2e} (true: {b_true:.2e})")

In [ ]:
# Visualize parameter uncertainty and verify fit
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Joint parameter distribution
axes[0].scatter(a_est, b_est * 1e4, alpha=0.3, s=10)
axes[0].scatter([a_true], [b_true * 1e4], c='red', s=200, marker='*',
                edgecolors='black', label='True values', zorder=5)
axes[0].set_xlabel('a (Pa·m⁶/mol²)')
axes[0].set_ylabel('b (×10⁻⁴ m³/mol)')
axes[0].set_title('Parameter Uncertainty')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Verify fit by comparing P-V curves
a_mean, b_mean = a_est.mean(), b_est.mean()
V_test = np.linspace(1e-4, 5e-4, 100)
T_test = 400  # K

P_true_curve = vdw_pressure(V_test, T_test, a_true, b_true) / 1e6
P_est_curve = vdw_pressure(V_test, T_test, a_mean, b_mean) / 1e6

axes[1].plot(V_test * 1e4, P_true_curve, 'b-', lw=2, label='True EOS')
axes[1].plot(V_test * 1e4, P_est_curve, 'r--', lw=2, label='Estimated EOS')
axes[1].set_xlabel('V (×10⁻⁴ m³/mol)')
axes[1].set_ylabel('P (MPa)')
axes[1].set_title(f'EOS Comparison at T = {T_test} K')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compute prediction error
P_error = np.abs(P_true_curve - P_est_curve).mean()
print(f"Mean absolute pressure error: {P_error:.4f} MPa")